# Spotify Matching Prep

This notebook prepares a match-ready dataset for the Spotify vs user-owned playlist matching exercise.

## Workflow
1. Load and minimally clean the playlist data.
2. Create short content documents from genre, mood, and title-token fields.
3. Generate real local text embeddings with `all-MiniLM-L6-v2`.
4. Reduce the embedding space with PCA.
5. Export one CSV for the R `MatchIt` script.

## 1. Setup

In [1]:
from pathlib import Path
import ast
import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA


def resolve_path(filename):
    candidates = [
        Path('interview_prep/data_science_assignment') / filename,
        Path(filename),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(filename)

DATA_PATH = resolve_path('playlist_revision_v05.txt')
OUTPUT_PATH = resolve_path('Spotify_Matching_Prep.ipynb').with_name('spotify_matching_input.csv')
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
N_CONTENT_PCS = 5

## 2. Load And Clean The Core Fields

This keeps the prep intentionally light:
- remove exact duplicate rows
- convert `'-'` placeholders to missing values
- create the Spotify-owned indicator
- parse the pre-existing `tokens` column into Python lists

In [2]:
def parse_tokens(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x).strip().lower() for x in parsed if str(x).strip()]
    except Exception:
        pass
    cleaned = re.sub(r'^\[|\]$', '', str(value))
    cleaned = cleaned.replace('"', '')
    return [x.strip().lower() for x in cleaned.split(',') if x.strip()]


def clean_text_value(value):
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    if not text:
        return None
    return text.replace('&', 'and').replace(' ', '_')


df = pd.read_csv(DATA_PATH, sep='\t')
df = df.drop_duplicates().copy()

df['genre_1'] = df['genre_1'].replace('-', pd.NA)
df['genre_2'] = df['genre_2'].replace('-', pd.NA)
df['genre_3'] = df['genre_3'].replace('-', pd.NA)
df['mood_1'] = df['mood_1'].replace('-', pd.NA)
df['mood_2'] = df['mood_2'].replace('-', pd.NA)
df['mood_3'] = df['mood_3'].replace('-', pd.NA)

df['is_spotify_owned'] = (df['owner'] == 'spotify').astype(int)
df['segment'] = np.where(df['is_spotify_owned'] == 1, 'Spotify-owned', 'User-owned')
df['non_owner_monthly_stream30s'] = (df['monthly_stream30s'] - df['monthly_owner_stream30s']).clip(lower=0)
df['token_list'] = df['tokens'].apply(parse_tokens)

spotify_genres = set(df.loc[df['is_spotify_owned'] == 1, 'genre_1'].dropna())
user_genres = set(df.loc[df['is_spotify_owned'] == 0, 'genre_1'].dropna())
shared_genres = spotify_genres & user_genres

df = df[df['genre_1'].isin(shared_genres)].copy()

df.groupby('segment').size()

segment
Spotify-owned       399
User-owned       413368
dtype: int64

## 3. Build Short Content Documents

Each short document includes:
- `genre_1`, `genre_2`, `genre_3`
- `mood_1`, `mood_2`, `mood_3`
- title `tokens`

This gives the embedding model a structured description of playlist content rather than title tokens alone.

In [3]:
def build_content_document(row):
    parts = []

    genre1 = clean_text_value(row['genre_1'])
    genre2 = clean_text_value(row['genre_2'])
    genre3 = clean_text_value(row['genre_3'])
    mood1 = clean_text_value(row['mood_1'])
    mood2 = clean_text_value(row['mood_2'])
    mood3 = clean_text_value(row['mood_3'])
    tokens = [clean_text_value(tok) for tok in row['token_list']]
    tokens = [tok for tok in tokens if tok is not None]

    if genre1:
        parts.append(f'genre1: {genre1}')
    if genre2:
        parts.append(f'genre2: {genre2}')
    if genre3:
        parts.append(f'genre3: {genre3}')
    if mood1:
        parts.append(f'mood1: {mood1}')
    if mood2:
        parts.append(f'mood2: {mood2}')
    if mood3:
        parts.append(f'mood3: {mood3}')
    if tokens:
        parts.append('tokens: ' + ' '.join(tokens))

    return '\n'.join(parts)


df['content_document'] = df.apply(build_content_document, axis=1)

df[['genre_1', 'genre_2', 'genre_3', 'mood_1', 'mood_2', 'mood_3', 'token_list', 'content_document']].head()

,genre_1,genre_2,genre_3,mood_1,mood_2,mood_3,token_list,content_document
0,Electronica,Alternative,Dance & House,Sensual,Excited,Brooding,[eclectronic],genre1: electronica\ngenre2: alternative\ngenr...
1,Pop,Latin,Rap,Excited,Energizing,Defiant,"[party, woodside]",genre1: pop\ngenre2: latin\ngenre3: rap\nmood1...
2,Alternative,Electronica,Pop,Excited,Aggressive,Urgent,"[five, hours]",genre1: alternative\ngenre2: electronica\ngenr...
3,Alternative,Rap,Pop,Defiant,Excited,Energizing,[summertime],genre1: alternative\ngenre2: rap\ngenre3: pop\...
4,Pop,R&B,Blues,Energizing,Lively,Fiery,"[latino, greatest, hits]",genre1: pop\ngenre2: randb\ngenre3: blues\nmoo...


## 4. Generate Local MiniLM Embeddings

This step uses the local sentence-transformer model `all-MiniLM-L6-v2`.

- Each playlist document is embedded into a dense numeric vector.
- The raw embedding is 384-dimensional.
- We will reduce it with PCA before matching.

In [4]:
embedding_model = SentenceTransformer(MODEL_NAME)
embedding_matrix = embedding_model.encode(
    df['content_document'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,
)

print('Embedding matrix shape:', embedding_matrix.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1617 [00:00<?, ?it/s]

Embedding matrix shape: (413767, 384)


## 5. Reduce The Embedding Space With PCA

The raw MiniLM embedding is dense and high-dimensional, so we compress it into a few continuous components.

These `content_pc` columns are what we will pass into `MatchIt` in R.

In [5]:
pca = PCA(n_components=N_CONTENT_PCS, svd_solver='randomized', random_state=42)
content_pcs = pca.fit_transform(embedding_matrix)

for i in range(N_CONTENT_PCS):
    df[f'content_pc{i + 1}'] = content_pcs[:, i]

pca_summary = pd.DataFrame({
    'component': [f'content_pc{i + 1}' for i in range(N_CONTENT_PCS)],
    'explained_variance_ratio': pca.explained_variance_ratio_,
    'cumulative_explained_variance': np.cumsum(pca.explained_variance_ratio_),
})

pca_summary

,component,explained_variance_ratio,cumulative_explained_variance
0,content_pc1,0.120135,0.120135
1,content_pc2,0.088125,0.208261
2,content_pc3,0.051730,0.259991
3,content_pc4,0.047244,0.307235
4,content_pc5,0.046611,0.353846


## 6. Export The Match-Ready Dataset

The exported CSV contains:
- the treatment flag
- the exact-matching field `genre_1`
- the structural covariates
- the `content_pc` variables from the local embedding step
- the outcome columns for later comparison

In [6]:
match_df = df[[
    'playlist_uri',
    'owner',
    'segment',
    'is_spotify_owned',
    'genre_1',
    'monthly_stream30s',
    'non_owner_monthly_stream30s',
    'n_tracks',
    'n_local_tracks',
    'n_artists',
    'n_albums',
    'content_document',
] + [f'content_pc{i + 1}' for i in range(N_CONTENT_PCS)]].copy()

match_df['log_n_tracks'] = np.log1p(match_df['n_tracks'])
match_df['log_n_local_tracks'] = np.log1p(match_df['n_local_tracks'])
match_df['log_n_artists'] = np.log1p(match_df['n_artists'])
match_df['log_n_albums'] = np.log1p(match_df['n_albums'])
match_df['log_monthly_stream30s'] = np.log1p(match_df['monthly_stream30s'])
match_df['log_non_owner_monthly_stream30s'] = np.log1p(match_df['non_owner_monthly_stream30s'])

match_df.to_csv(OUTPUT_PATH, index=False)

print(f'Wrote match-ready data to: {OUTPUT_PATH}')
print(f'Rows: {len(match_df):,}')
match_df.groupby('segment').size()

Wrote match-ready data to: spotify_matching_input.csv
Rows: 413,767


segment
Spotify-owned       399
User-owned       413368
dtype: int64

In [7]:
preview_cols = [
    'segment', 'genre_1', 'content_document',
    'content_pc1', 'content_pc2', 'content_pc3', 'content_pc4', 'content_pc5'
]
match_df[preview_cols].head()

,segment,genre_1,content_document,content_pc1,content_pc2,content_pc3,content_pc4,content_pc5
0,User-owned,Electronica,genre1: electronica\ngenre2: alternative\ngenr...,-0.100893,-0.003736,-0.047542,0.239091,0.150305
1,User-owned,Pop,genre1: pop\ngenre2: latin\ngenre3: rap\nmood1...,0.210433,-0.012017,-0.026363,0.028814,-0.035884
2,User-owned,Alternative,genre1: alternative\ngenre2: electronica\ngenr...,0.014799,0.019014,-0.090451,0.007942,0.211079
3,User-owned,Alternative,genre1: alternative\ngenre2: rap\ngenre3: pop\...,0.151983,-0.071713,0.035639,-0.091186,0.030501
4,User-owned,Pop,genre1: pop\ngenre2: randb\ngenre3: blues\nmoo...,0.093026,0.106397,0.017237,-0.009315,-0.073256
